# ➰ Plotting Probabilities of Corrupted Model Runs ➰

## Parameters

In [1]:
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"

oneshot_template = ""
last_prompt_template = ""
num_shots = 1

data_path = "data/modal-verbs.csv"
src_lang_base = "eng"
tgt_lang_base = "tur"
src_lang_source = "eng"
tgt_lang_source = "ger"

sample_size = 5
random_seed = 42

sentences_src_prefix_base = "phrase-"
sentences_tgt_prefix_base = "phrase_cutoff_after_subject-"
sentences_src_prefix_source = "phrase-"
sentences_tgt_prefix_source = "phrase_cutoff_after_subject-"

modal_base_prefix = "modal-"
verb_base_prefix = "verb-"
modal_source_prefix = "modal-"
verb_source_prefix = "verb-"

load_data = False # CRUCIAL: if True, doesn't run intervention, just loads existing data


block_intervention_save_path = None
head_intervention_save_path = None

hf_token = None

In [2]:
# Parameters
block_intervention_save_path = "output/intervention/probs/block_modal-verbs_mgpt_eng-tur_eng-ger.csv"
head_intervention_save_path = "output/intervention/probs/head_modal-verbs_mgpt_eng-tur_eng-ger.csv"
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"
oneshot_template = "{lang_src}: \"{sentence_src}\" - {lang_tgt}: \"{sentence_tgt}\""
last_prompt_template = "{lang_src}: \"{sentence_src}\" - {lang_tgt}: \"{sentence_tgt}"
num_shots = 1
data_path = "data/modal-verbs.csv"
src_lang_base = "eng"
tgt_lang_base = "tur"
src_lang_source = "eng"
tgt_lang_source = "ger"
sample_size = 5
random_seed = 42
sentences_src_prefix_base = "phrase-"
sentences_tgt_prefix_base = "phrase_cutoff_after_subject-"
sentences_src_prefix_source = "phrase-"
sentences_tgt_prefix_source = "phrase_cutoff_after_subject-"
modal_base_prefix = "modal-"
verb_base_prefix = "verb-"
modal_source_prefix = "modal-"
verb_source_prefix = "verb-"


In [3]:
if block_intervention_save_path is None:
    block_intervention_save_path = \
        f"output/intervention/block-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"
if head_intervention_save_path is None:
    head_intervention_save_path = \
        f"output/intervention/head-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"
block_intervention_plot_save_path = block_intervention_save_path.replace(".csv", ".png")
head_intervention_plot_save_path = head_intervention_save_path.replace(".csv", ".png")

shot_data_src_base = f'phrase-{src_lang_base}'
shot_data_tgt_base = f'phrase-{tgt_lang_base}'
shot_data_src_source = f'phrase-{src_lang_source}'
shot_data_tgt_source = f'phrase-{tgt_lang_source}'

## Setup

In [4]:
import torch
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from huggingface_hub import login
from typing import List, Dict, Any
from statsmodels.formula.api import mixedlm
from transformers import AutoModelForCausalLM, AutoTokenizer

import pyvene as pv
from pyvene import embed_to_distrib, top_vals, format_token
from pyvene.models.modeling_utils import getattr_for_torch_module

from create_datasets.parallel_dataset import ParallelDataset
from utils.model_args import model_to_num_layers_attr, model_to_num_heads_attr

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
sm = torch.nn.Softmax(dim=2)

login(hf_token)

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


### Set up the Model

In [5]:
if not load_data:
    model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

    num_layers = getattr_for_torch_module(model, model_to_num_layers_attr[model_id])
    num_heads = getattr_for_torch_module(model, model_to_num_heads_attr[model_id])

### Set up the Data

In [6]:
if not load_data:
    df = pd.read_csv(data_path)

    base_df = df.sample(n=sample_size, random_state=random_seed).reset_index(drop=True)
    source_df = base_df.sample(n=sample_size, random_state=random_seed).reset_index(drop=True) # shuffled base

    base_dataset = ParallelDataset(
        model_id,
        dataframe=base_df,
        lang_src=src_lang_base,
        lang_tgt=tgt_lang_base,
        sentences_src_prefix=sentences_src_prefix_base,
        sentences_tgt_prefix=sentences_tgt_prefix_base,
        random_seed=random_seed,
    )

    source_dataset = ParallelDataset(
        model_id,
        dataframe=source_df,
        lang_src=src_lang_source,
        lang_tgt=tgt_lang_source,
        sentences_src_prefix=sentences_src_prefix_source,
        sentences_tgt_prefix=sentences_tgt_prefix_source,
        random_seed=random_seed,
    )

    base_prompts = base_dataset.format(
        oneshot_template,
        shots=num_shots,
        last_prompt_template=last_prompt_template,
        shot_data_src=shot_data_src_base,
        shot_data_tgt=shot_data_tgt_base,
        shuffle_shots=False,
        )
    print("=====Example of Base Prompt=====")
    print(base_prompts[0])

    base_tokens = base_dataset.prompts_to_tokens()

    source_prompts = source_dataset.format(
        oneshot_template,
        shots=num_shots,
        last_prompt_template=last_prompt_template,
        shot_data_src=shot_data_src_source,
        shot_data_tgt=shot_data_tgt_source,
        shuffle_shots=False,
        )
    print("=====Example of Source Prompt=====")
    print(source_prompts[0])

    source_tokens = source_dataset.prompts_to_tokens()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

=====Example of Base Prompt=====
English: "the spider has to die" - Türkçe: "örümcek ölmek zorunda"
English: "the mathematician has to count" - Türkçe: "matematikçi
=====Example of Source Prompt=====
English: "the man has to work" - Deutsch: "der Mann muss arbeiten"
English: "the spider has to die" - Deutsch: "die Spinne


### Functions

#### Intervention Config

In [7]:
def intervention_config(model_type, intervention_type, unit, layer):
    """
    Parameters
    __________

    model_type: model type
    intervention_type: component for RepresentationConfig, e.g. head_attention_value_output
    unit: string to define the component type, e.g. "h" (head), "pos" (position), "h.pos" (head within position)
    later: layer id
    """
    # Set up the config to intervene
    config = pv.IntervenableConfig(
        model_type=model_type,
        representations=[
            pv.RepresentationConfig(
                layer,  # layer
                intervention_type,  # intervention type
                unit,  # intervention unit is now [pos] within [h]
                1,  # max number of unit
            ),
        ],
        intervention_types=pv.VanillaIntervention,
    )
    return config

#### Intervention Function

In [ ]:
def intervention_data(
        base: torch.Tensor,
        source: torch.Tensor,
        base_pos: int,
        source_pos: int,
        tokentype2token: Dict[str, str],
        component_type: str,
        sentence_index: int = None,
        head_i: int = None,
        data: List[Dict[str, Any]] = None
    ) -> List[Dict[str, Any]]:
    """
    Collect intervention data for a given model component (block output or head attention value output).

    Parameters
    ----------
    base : torch.Tensor
        The tokenized base prompt tensor.
    source : torch.Tensor
        The tokenized source prompt tensor.
    base_pos : int
        The position of the last token in the base prompt.
    source_pos : int
        The position of the last token in the source prompt.
    tokentype2token : Dict[str, str]
        A dictionary mapping token types (e.g., 'noun-base', 'adj-base') to their corresponding token strings.
    component_type : str
        The type of model component to intervene on. Must be either 'block_output' or 'head_attention_value_output'.
    head_i : int, optional
        The index of the head to intervene on (if applicable). Only used for head-level interventions.
    data : List[Dict[str, Any]], optional
        A list to append the collected data to. If None, a new list will be created.

    Returns
    -------
    List[Dict[str, Any]]
        A list of dictionaries containing the intervention data, with keys:
        - "token_type": The type of token (e.g., 'noun-base', 'adj-base').
        - "token": The token string.
        - "prob": The probability of the token after intervention.
        - "layer": The layer index.
        - "head_i": The head index (if applicable).
        - "pos": The position index.
        - "type": The component type (e.g., 'block_output').

    Raises
    ------
    NotImplementedError
        If the component_type is not 'block_output' or 'head_attention_value_output'.
    """
    for token_type, token in tokentype2token.items():
        tokentype2token[token_type] = " "+token
    if data is None:
        data = []

    # print("TOKENS AT HAND:")
    # for token_type, token in tokentype2token.items():
    #     token_id = tokenizer.encode(token, add_special_tokens=False)[0]
    #     print(f"{token}\t{token_id}\t{tokenizer.convert_ids_to_tokens([token_id])[0]}")
    
    for layer_i in range(num_layers):
        if component_type == "block_output":
            unit = "pos"
        elif component_type == "head_attention_value_output":
            unit = "h.pos"
        else:
            raise NotImplementedError("Unsupported component type: {component_type}")
        config = intervention_config(
            type(model), component_type, unit, layer_i
        )
        intervenable = pv.IntervenableModel(config, model)
        if head_i is not None:
            unit_locations = {
                "sources->base": (
                    [[[[head_i]], [[source_pos]]]],  # intervene w/ target_head's pos_i
                    [[[[head_i]], [[base_pos]]]]
                ),
            }
        else:
            unit_locations = {"sources->base": (source_pos, base_pos)}
        # print(unit_locations)
        # print(base)
        # print(source)
        _, counterfactual_outputs = intervenable(
            base,
            source,
            unit_locations,
        )
        with torch.inference_mode():
            distrib = sm(counterfactual_outputs.logits)
        # if layer_i in [0, 1, 13, 14, 15, 16, num_layers - 2, num_layers - 1]:
        #     print(f"\nINTERVENTION AT LAYER #{layer_i}:")
        #     # top_toks = tokenizer.convert_ids_to_tokens(distrib[0][base_pos].topk(10).indices.tolist())
        #     top_toks = distrib[0][base_pos].topk(10)
        #     print(F"TOP 10 TOKENS:")
        #     for tok_place_i, tok in enumerate(top_toks.indices):
        #         print(f"{tok}\t{tokenizer.convert_ids_to_tokens([tok])[0]}\t{top_toks.values[tok_place_i].item()}")
        #     print("TOKENS AT HAND:")
        #     for token_type, token in tokentype2token.items():
        #         token_id = tokenizer.encode(token, add_special_tokens=False)[0]
        #         print(f"{token_id}\t{tokenizer.convert_ids_to_tokens([token_id])[0]}\t{distrib[0][base_pos][token_id]}\t{token}")
        for token_type, token in tokentype2token.items():
            # print(token_type, token, tokenizer.encode(token, add_special_tokens=False)[0], tokenizer.convert_ids_to_tokens(tokenizer.encode(token, add_special_tokens=False)[0]))
            data.append(
                {
                    "sentence_id": sentence_index,
                    "token_type": token_type,
                    "token": token,
                    "prob": float(distrib[0][base_pos][tokenizer.encode(token, add_special_tokens=False)[0],]),
                    "layer": layer_i,
                    "head_id": head_i,
                    "pos": base_pos,
                    "type": component_type,
                }
            )
    return data

## Block Output Intervention

### Calculating

In [9]:
if load_data:
    df = pd.read_csv(block_intervention_save_path)

else:
    data = []
    source_df_list = list(source_df.iterrows())
    for row_i, row in base_dataset.df.iterrows():
        # tokenize prompts
        prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)

        prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
        # last token index
        pos_base = prompt_base.input_ids.size(1) - 1
        pos_source = prompt_source.input_ids.size(1) - 1
        # token type to token dict
        tokentype2token = {
            f"modal-base-{tgt_lang_base}": row[f'{modal_base_prefix}{tgt_lang_base}'],
            f"verb-base-{tgt_lang_base}": row[f'{verb_base_prefix}{tgt_lang_base}'],
            f"modal-source-{tgt_lang_base}": source_df_list[row_i][1][f'{modal_source_prefix}{tgt_lang_base}'],
            f"verb-source-{tgt_lang_base}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_base}'],

            f"modal-base-{tgt_lang_source}": row[f'{modal_base_prefix}{tgt_lang_source}'],
            f"verb-base-{tgt_lang_source}": row[f'{verb_base_prefix}{tgt_lang_source}'],
            f"modal-source-{tgt_lang_source}": source_df_list[row_i][1][f'{modal_source_prefix}{tgt_lang_source}'],
            f"verb-source-{tgt_lang_source}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_source}'],

            f"modal-base-{src_lang_base}": row[f'{modal_base_prefix}{src_lang_base}'],
            f"verb-base-{src_lang_base}": row[f'{verb_base_prefix}{src_lang_base}'],
            f"modal-source-{src_lang_base}": source_df_list[row_i][1][f'{modal_source_prefix}{src_lang_base}'],
            f"verb-source-{src_lang_base}": source_df_list[row_i][1][f'{verb_source_prefix}{src_lang_base}'],
        }
        # print(tokentype2token)

        data = intervention_data(
            prompt_base, 
            prompt_source, 
            pos_base,
            pos_source,
            tokentype2token,
            sentence_index=row_i,
            component_type="block_output",
            data=data
        )
    df = pd.DataFrame(data)
    df.to_csv(block_intervention_save_path)

TOKENS AT HAND:
 zorunda	34867	Ġzor
 saymak	9288	Ġsay
 zorunda	34867	Ġzor
 ölmek	30448	ĠÃ¶l
 muss	9830	Ġmuss
 rechnen	29096	Ġrech
 muss	9830	Ġmuss
 sterben	12723	Ġster
 has to	1296	Ġhas
 count	7637	Ġcount
 has to	1296	Ġhas
 die	547	Ġdie


TOKENS AT HAND:
 zorunda	34867	Ġzor
 ölmek	30448	ĠÃ¶l
 zorunda	34867	Ġzor
 çalışmak	26217	ĠÃ§alÄ±ÅŁ
 muss	9830	Ġmuss
 sterben	12723	Ġster
 muss	9830	Ġmuss
 arbeiten	44598	Ġarbeiten
 has to	1296	Ġhas
 die	547	Ġdie
 has to	1296	Ġhas
 work	2442	Ġwork


TOKENS AT HAND:
 zorunda	34867	Ġzor
 uyumak	27014	Ġuy
 zorunda	34867	Ġzor
 uyumak	27014	Ġuy
 muss	9830	Ġmuss
 schlafen	1244	Ġsch
 muss	9830	Ġmuss
 schlafen	1244	Ġsch
 has to	1296	Ġhas
 sleep	54667	Ġsleep
 has to	1296	Ġhas
 sleep	54667	Ġsleep


TOKENS AT HAND:
 zorunda	34867	Ġzor
 koşmak	4443	Ġko
 zorunda	34867	Ġzor
 saymak	9288	Ġsay
 muss	9830	Ġmuss
 rennen	400	Ġr
 muss	9830	Ġmuss
 rechnen	29096	Ġrech
 has to	1296	Ġhas
 run	4560	Ġrun
 has to	1296	Ġhas
 count	7637	Ġcount


TOKENS AT HAND:
 zorunda	34867	Ġzor
 çalışmak	26217	ĠÃ§alÄ±ÅŁ
 zorunda	34867	Ġzor
 koşmak	4443	Ġko
 muss	9830	Ġmuss
 arbeiten	44598	Ġarbeiten
 muss	9830	Ġmuss
 rennen	400	Ġr
 has to	1296	Ġhas
 work	2442	Ġwork
 has to	1296	Ġhas
 run	4560	Ġrun


### Plotting

In [10]:
# Ensure the 'prob' column is numeric
df['prob'] = pd.to_numeric(df['prob'], errors='coerce')

# Drop rows with NaN in 'prob'
df = df.dropna(subset=['prob'])

# Create a line plot for token probabilities over layers
fig = px.line(
    df.groupby(['token_type', 'layer'], as_index=False)['prob'].mean(),
    x="layer",
    y="prob",
    color="token_type",
    title=f"Probabilities after Block Intervention ({src_lang_base}-{tgt_lang_base} base & {src_lang_source}-{tgt_lang_source} source)",
    labels={"layer": "Layer", "prob": "Probability", "token_type": "Token"},
    # category_orders={"layer": [str(i) for i in range(model.config.n_layer)]},
)

# Show the plot
fig.show()
# fig.write_image(block_intervention_plot_save_path)

## Head Intervention

### Calculating

In [11]:
if load_data:
    df = pd.read_csv(head_intervention_save_path)
else:
    data = []
    source_df_list = list(source_df.iterrows())

    for row_i, row in base_dataset.df.iterrows():
        # tokenize prompts
        prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)
        prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
        # last token index
        pos_base = prompt_base.input_ids.size(1) - 1
        pos_source = prompt_source.input_ids.size(1) - 1
        # token type to token dict
        tokentype2token = {
            f"modal-base-{tgt_lang_base}": row[f'{modal_base_prefix}{tgt_lang_base}'],
            f"verb-base-{tgt_lang_base}": row[f'{verb_base_prefix}{tgt_lang_base}'],
            f"modal-source-{tgt_lang_base}": source_df_list[row_i][1][f'{modal_source_prefix}{tgt_lang_base}'],
            f"verb-source-{tgt_lang_base}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_base}'],

            f"modal-base-{tgt_lang_source}": row[f'{modal_base_prefix}{tgt_lang_source}'],
            f"verb-base-{tgt_lang_source}": row[f'{verb_base_prefix}{tgt_lang_source}'],
            f"modal-source-{tgt_lang_source}": source_df_list[row_i][1][f'{modal_source_prefix}{tgt_lang_source}'],
            f"verb-source-{tgt_lang_source}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_source}'],

            f"modal-base-{src_lang_base}": row[f'{modal_base_prefix}{src_lang_base}'],
            f"verb-base-{src_lang_base}": row[f'{verb_base_prefix}{src_lang_base}'],
            f"modal-source-{src_lang_base}": source_df_list[row_i][1][f'{modal_source_prefix}{src_lang_base}'],
            f"verb-source-{src_lang_base}": source_df_list[row_i][1][f'{verb_source_prefix}{src_lang_base}'],
        }

        for head_i in range(num_heads):
            data = intervention_data(
                prompt_base, 
                prompt_source, 
                pos_base,
                pos_source,
                tokentype2token, 
                "head_attention_value_output", 
                sentence_index=row_i,
                head_i=head_i,
                data=data,
            )
    df = pd.DataFrame(data)
    df.to_csv(head_intervention_save_path)

TOKENS AT HAND:
 zorunda	34867	Ġzor
 saymak	9288	Ġsay
 zorunda	34867	Ġzor
 ölmek	30448	ĠÃ¶l
 muss	9830	Ġmuss
 rechnen	29096	Ġrech
 muss	9830	Ġmuss
 sterben	12723	Ġster
 has to	1296	Ġhas
 count	7637	Ġcount
 has to	1296	Ġhas
 die	547	Ġdie


TOKENS AT HAND:
  zorunda	227	Ġ
  saymak	227	Ġ
  zorunda	227	Ġ
  ölmek	227	Ġ
  muss	227	Ġ
  rechnen	227	Ġ
  muss	227	Ġ
  sterben	227	Ġ
  has to	227	Ġ
  count	227	Ġ
  has to	227	Ġ
  die	227	Ġ


TOKENS AT HAND:
   zorunda	227	Ġ
   saymak	227	Ġ
   zorunda	227	Ġ
   ölmek	227	Ġ
   muss	227	Ġ
   rechnen	227	Ġ
   muss	227	Ġ
   sterben	227	Ġ
   has to	227	Ġ
   count	227	Ġ
   has to	227	Ġ
   die	227	Ġ


TOKENS AT HAND:
    zorunda	227	Ġ
    saymak	227	Ġ
    zorunda	227	Ġ
    ölmek	227	Ġ
    muss	227	Ġ
    rechnen	227	Ġ
    muss	227	Ġ
    sterben	227	Ġ
    has to	227	Ġ
    count	227	Ġ
    has to	227	Ġ
    die	227	Ġ


TOKENS AT HAND:
     zorunda	227	Ġ
     saymak	227	Ġ
     zorunda	227	Ġ
     ölmek	227	Ġ
     muss	227	Ġ
     rechnen	227	Ġ
     muss	227	Ġ
     sterben	227	Ġ
     has to	227	Ġ
     count	227	Ġ
     has to	227	Ġ
     die	227	Ġ


...

TOKENS AT HAND:
            zorunda	227	Ġ
            çalışmak	227	Ġ
            zorunda	227	Ġ
            koşmak	227	Ġ
            muss	227	Ġ
            arbeiten	227	Ġ
            muss	227	Ġ
            rennen	227	Ġ
            has to	227	Ġ
            work	227	Ġ
            has to	227	Ġ
            run	227	Ġ


TOKENS AT HAND:
             zorunda	227	Ġ
             çalışmak	227	Ġ
             zorunda	227	Ġ
             koşmak	227	Ġ
             muss	227	Ġ
             arbeiten	227	Ġ
             muss	227	Ġ
             rennen	227	Ġ
             has to	227	Ġ
             work	227	Ġ
             has to	227	Ġ
             run	227	Ġ


TOKENS AT HAND:
              zorunda	227	Ġ
              çalışmak	227	Ġ
              zorunda	227	Ġ
              koşmak	227	Ġ
              muss	227	Ġ
              arbeiten	227	Ġ
              muss	227	Ġ
              rennen	227	Ġ
              has to	227	Ġ
              work	227	Ġ
              has to	227	Ġ
              run	227	Ġ


TOKENS AT HAND:
               zorunda	227	Ġ
               çalışmak	227	Ġ
               zorunda	227	Ġ
               koşmak	227	Ġ
               muss	227	Ġ
               arbeiten	227	Ġ
               muss	227	Ġ
               rennen	227	Ġ
               has to	227	Ġ
               work	227	Ġ
               has to	227	Ġ
               run	227	Ġ


TOKENS AT HAND:
                zorunda	227	Ġ
                çalışmak	227	Ġ
                zorunda	227	Ġ
                koşmak	227	Ġ
                muss	227	Ġ
                arbeiten	227	Ġ
                muss	227	Ġ
                rennen	227	Ġ
                has to	227	Ġ
                work	227	Ġ
                has to	227	Ġ
                run	227	Ġ


### Plotting

In [12]:
# print(df.head())

# Ensure the 'prob' column is numeric
df['prob'] = pd.to_numeric(df['prob'], errors='coerce')

# Drop rows with NaN in 'prob'
df = df.dropna(subset=['prob'])

# Iterate over the unique tokens and create a heatmap for each
tokens = df["token_type"].unique()
for token in tokens:
    token_df = df[df["token_type"] == token].groupby(['layer', 'head_id'], as_index=False).mean(numeric_only=True).reset_index()
    heatmap_data = token_df.pivot(index="layer", columns="head_id", values="prob")

    fig = px.imshow(
        heatmap_data,
        labels={"x": "Head", "y": "Layer", "color": "Probability"},
        title=f"Probability Heatmap for Token after Head Intervention: {token}",
        color_continuous_scale="viridis",
    )
    fig.show()

In [13]:
# HEAD 15.2 PREFERS NOUN-SOURCE MORE THAN ADJ-SOURCE?
# (difference not too big but it's interesting it is more visible. It sees the adjective too though.)

# Next up: Try with other languages as well (English-Italian, German-French)
# Bonus: Look into h.pos vs. pos: wtf are they doing?